In [1]:

# Step 1: Loading the Dataset
# I uploaded the original ACS customer churn dataset.
# I loaded it using Pandas and checked its structure.

import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("Dataset_ATS_v2.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())


Saving Dataset_ATS_v2.csv to Dataset_ATS_v2 (1).csv
Dataset shape: (7043, 10)

Column names:
['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']

First five records:


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


In [2]:

# Step 2: Understanding the Dataset
# I checked the dataset structure, data types, and
# basic statistics before starting preprocessing.

print("Dataset information:")
df.info()

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary:")
display(df.describe(include="all"))


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   gender           7043 non-null   object
 1   SeniorCitizen    7043 non-null   int64 
 2   Dependents       7043 non-null   object
 3   tenure           7043 non-null   int64 
 4   PhoneService     7043 non-null   object
 5   MultipleLines    7043 non-null   object
 6   InternetService  7043 non-null   object
 7   Contract         7043 non-null   object
 8   MonthlyCharges   7043 non-null   int64 
 9   Churn            7043 non-null   object
dtypes: int64(3), object(7)
memory usage: 550.4+ KB

Data types:
gender             object
SeniorCitizen       int64
Dependents         object
tenure              int64
PhoneService       object
MultipleLines      object
InternetService    object
Contract           object
MonthlyCharges      int64
Churn              object
dtype: 

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
count,7043,7043.000000,7043,7043.000000,7043,7043,7043,7043,7043.000000,7043
unique,2,NaN,2,NaN,2,2,2,3,NaN,2
top,Male,NaN,No,NaN,Yes,No,DSL,Month-to-month,NaN,No
freq,3555,NaN,4933,NaN,6361,4072,3947,3875,NaN,5174
mean,NaN,0.162147,NaN,32.371149,NaN,NaN,NaN,NaN,64.758768,NaN
std,NaN,0.368612,NaN,24.559481,NaN,NaN,NaN,NaN,30.091650,NaN
min,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,NaN,18.000000,NaN
25%,NaN,0.000000,NaN,9.000000,NaN,NaN,NaN,NaN,36.000000,NaN
50%,NaN,0.000000,NaN,29.000000,NaN,NaN,NaN,NaN,70.000000,NaN
75%,NaN,0.000000,NaN,55.000000,NaN,NaN,NaN,NaN,90.000000,NaN


In [3]:

# Step 3: Data Quality Assessment
# I checked missing values, duplicate records, blank
# text values, and invalid numerical values.

missing = df.isnull().sum()
duplicates = df.duplicated().sum()

text_cols = df.select_dtypes(include="object").columns
blank_values = df[text_cols].apply(
    lambda col: col.str.strip().eq("").sum()
)

print("Missing values:", missing.sum())
print("Duplicate records:", duplicates)
print("Blank text values:", blank_values.sum())
print("Negative tenure:", (df["tenure"] < 0).sum())
print("Zero tenure:", (df["tenure"] == 0).sum())
print("Negative monthly charges:",
      (df["MonthlyCharges"] < 0).sum())


Missing values: 0
Duplicate records: 302
Blank text values: 0
Negative tenure: 0
Zero tenure: 11
Negative monthly charges: 0


In [4]:

# Step 4: Categorical Data Validation
# I checked the unique categories and identified
# inconsistent phone-service information.

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"{col}: {df[col].unique().tolist()}")

conflicting_rows = (
    (df["PhoneService"] == "No") &
    (df["MultipleLines"] == "Yes")
).sum()

print("\nConflicting phone-service records:", conflicting_rows)
print("Total records retained:", len(df))


gender: ['Female', 'Male']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No', 'Yes']
InternetService: ['DSL', 'Fiber optic']
Contract: ['Month-to-month', 'One year', 'Two year']
Churn: ['Yes', 'No']

Conflicting phone-service records: 260
Total records retained: 7043


In [5]:

# Step 5: Feature Selection and Train-Test Split
# I separated the input features from the Churn target.
# I split the dataset into 80% training and 20% testing.
# I used stratification to preserve the churn distribution.

from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Input features:", X.shape)
print("Target:", y.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts())

print("\nTesting churn distribution:")
print(y_test.value_counts())


Input features: (7043, 9)
Target: (7043,)
Training features: (5634, 9)
Testing features: (1409, 9)

Training churn distribution:
Churn
No     4139
Yes    1495
Name: count, dtype: int64

Testing churn distribution:
Churn
No     1035
Yes     374
Name: count, dtype: int64


In [6]:

# Step 6: Categorical Feature Encoding
# I converted categorical features into numerical values.
# I fitted the encoder on training data only to prevent
# data leakage and applied it to both datasets.

from sklearn.preprocessing import OneHotEncoder

cat_cols = X_train.select_dtypes(include="object").columns

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_encoded = encoder.fit_transform(X_train[cat_cols])
test_encoded = encoder.transform(X_test[cat_cols])

encoded_cols = encoder.get_feature_names_out(cat_cols)

print("Categorical columns:", len(cat_cols))
print("Encoded feature columns:", len(encoded_cols))
print("Training encoded shape:", train_encoded.shape)
print("Testing encoded shape:", test_encoded.shape)
print("\nEncoded feature names:")
print(encoded_cols.tolist())


Categorical columns: 6
Encoded feature columns: 13
Training encoded shape: (5634, 13)
Testing encoded shape: (1409, 13)

Encoded feature names:
['gender_Female', 'gender_Male', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year']


In [7]:

# Step 7: Numerical Feature Scaling
# I standardized tenure and MonthlyCharges using
# StandardScaler fitted only on the training data.
# I kept SeniorCitizen unchanged as a binary feature.

from sklearn.preprocessing import StandardScaler

num_cols = ["tenure", "MonthlyCharges"]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(X_train[num_cols])
test_scaled = scaler.transform(X_test[num_cols])

print("Scaled numerical features:", num_cols)
print("Training scaled shape:", train_scaled.shape)
print("Testing scaled shape:", test_scaled.shape)

print("\nTraining feature means after scaling:")
print(train_scaled.mean(axis=0).round(4))

print("\nTraining feature standard deviations:")
print(train_scaled.std(axis=0).round(4))


Scaled numerical features: ['tenure', 'MonthlyCharges']
Training scaled shape: (5634, 2)
Testing scaled shape: (1409, 2)

Training feature means after scaling:
[0. 0.]

Training feature standard deviations:
[1. 1.]


In [8]:

# Step 8: Combining Preprocessed Features
# I combined encoded categorical features, scaled numerical
# features, and the binary SeniorCitizen feature.
# I kept training and testing data separate.

feature_names = list(encoded_cols) + num_cols + ["SeniorCitizen"]

X_train_final = pd.DataFrame(
    np.column_stack([
        train_encoded,
        train_scaled,
        X_train["SeniorCitizen"].to_numpy()
    ]),
    columns=feature_names,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    np.column_stack([
        test_encoded,
        test_scaled,
        X_test["SeniorCitizen"].to_numpy()
    ]),
    columns=feature_names,
    index=X_test.index
)

print("Training dataset:", X_train_final.shape)
print("Testing dataset:", X_test_final.shape)
print("Total features:", len(feature_names))
print("Missing values:", X_train_final.isnull().sum().sum()
      + X_test_final.isnull().sum().sum())

display(X_train_final.head())


Training dataset: (5634, 16)
Testing dataset: (1409, 16)
Total features: 16
Missing values: 0


,gender_Female,gender_Male,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,tenure,MonthlyCharges,SeniorCitizen
3757,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,-1.276682,-0.149881,0.0
3165,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,-0.096089,-0.315125,0.0
4912,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.555273,1.172071,1.0
3877,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.433143,0.048412,0.0
3818,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,-1.276682,-1.471834,0.0


In [9]:

# Step 9: Preparing the Clustering Dataset
# I prepared a separate feature dataset for K-Means clustering.
# I encoded categorical variables and standardized numerical
# features without including the Churn target variable.

from sklearn.preprocessing import StandardScaler

X_cluster = pd.get_dummies(X, dtype=int)

cluster_scaler = StandardScaler()
X_cluster[num_cols] = cluster_scaler.fit_transform(
    X_cluster[num_cols]
)

print("Clustering dataset:", X_cluster.shape)
print("Missing values:", X_cluster.isnull().sum().sum())
print("Churn excluded:", "Churn" not in X_cluster.columns)

print("\nClustering feature names:")
print(X_cluster.columns.tolist())


Clustering dataset: (7043, 16)
Missing values: 0
Churn excluded: True

Clustering feature names:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'gender_Female', 'gender_Male', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year']


In [10]:

# Step 10: Exporting Preprocessed Data
# I saved the training, testing, and clustering datasets.
# I also saved the fitted encoder and scalers so my
# teammates can reuse the same preprocessing settings.

import os
import joblib

os.makedirs("Data_Preparation", exist_ok=True)

# Encode the target variable
target_mapping = {"No": 0, "Yes": 1}

train_export = X_train_final.copy()
test_export = X_test_final.copy()

train_export["Churn"] = y_train.map(target_mapping)
test_export["Churn"] = y_test.map(target_mapping)

# Save datasets
train_export.to_csv(
    "Data_Preparation/train_preprocessed.csv", index=False
)
test_export.to_csv(
    "Data_Preparation/test_preprocessed.csv", index=False
)
X_cluster.to_csv(
    "Data_Preparation/clustering_preprocessed.csv", index=False
)

# Save preprocessing objects
joblib.dump(encoder, "Data_Preparation/encoder.pkl")
joblib.dump(scaler, "Data_Preparation/predictive_scaler.pkl")
joblib.dump(cluster_scaler, "Data_Preparation/clustering_scaler.pkl")

# Verify saved files
for filename in sorted(os.listdir("Data_Preparation")):
    path = os.path.join("Data_Preparation", filename)
    print(filename, "-", os.path.getsize(path), "bytes")

print("\nTraining export:", train_export.shape)
print("Testing export:", test_export.shape)
print("Clustering export:", X_cluster.shape)


clustering_preprocessed.csv - 472931 bytes
clustering_scaler.pkl - 935 bytes
encoder.pkl - 2125 bytes
predictive_scaler.pkl - 935 bytes
test_preprocessed.csv - 136857 bytes
train_preprocessed.csv - 546508 bytes

Training export: (5634, 17)
Testing export: (1409, 17)
Clustering export: (7043, 16)


In [11]:

# Step 11: Backing Up Preprocessed Files
# I saved copies of my preprocessing deliverables
# to Google Drive to prevent data loss.

from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source = "/content/Data_Preparation"
destination = "/content/drive/MyDrive/ACS_WIL/Data_Preparation"

os.makedirs(destination, exist_ok=True)

for filename in os.listdir(source):
    shutil.copy2(
        os.path.join(source, filename),
        os.path.join(destination, filename)
    )

print("Google Drive backup completed!\n")

for filename in sorted(os.listdir(destination)):
    path = os.path.join(destination, filename)
    print(filename, "-", os.path.getsize(path), "bytes")


Mounted at /content/drive
Google Drive backup completed!

clustering_preprocessed.csv - 472931 bytes
clustering_scaler.pkl - 935 bytes
encoder.pkl - 2125 bytes
predictive_scaler.pkl - 935 bytes
test_preprocessed.csv - 136857 bytes
train_preprocessed.csv - 546508 bytes


In [12]:

# Step 12: Creating the Data Quality Report
# I documented the data-quality issues identified during
# inspection and explained why I retained the records.

quality_report = pd.DataFrame([
    ["Total records", 7043, "Retained all records"],
    ["Missing values", 0, "No treatment required"],
    ["Blank text values", 0, "No treatment required"],
    ["Duplicate-looking rows", 302,
     "Retained because no customer ID is available"],
    ["Zero tenure", 11,
     "Retained and documented"],
    ["Negative tenure", 0, "No treatment required"],
    ["Negative monthly charges", 0,
     "No treatment required"],
    ["Conflicting phone-service records", 260,
     "Retained because correction cannot be verified"]
], columns=["Check", "Count", "Decision"])

report_path = "Data_Preparation/data_quality_report.csv"
quality_report.to_csv(report_path, index=False)

# Back up the report to Google Drive
shutil.copy2(report_path, destination)

print("Data quality report saved successfully!")
print("Total report checks:", len(quality_report))
display(quality_report)


Data quality report saved successfully!
Total report checks: 8


,Check,Count,Decision
0,Total records,7043,Retained all records
1,Missing values,0,No treatment required
2,Blank text values,0,No treatment required
3,Duplicate-looking rows,302,Retained because no customer ID is available
4,Zero tenure,11,Retained and documented
5,Negative tenure,0,No treatment required
6,Negative monthly charges,0,No treatment required
7,Conflicting phone-service records,260,Retained because correction cannot be verified


In [13]:

# Step 13: Saving Feature Information and Configuration
# I documented the feature names, target encoding,
# preprocessing methods, and train-test split settings
# to support reproducibility and team collaboration.

import json

feature_info = {
    "original_features": X.columns.tolist(),
    "predictive_features": X_train_final.columns.tolist(),
    "clustering_features": X_cluster.columns.tolist(),
    "categorical_features": cat_cols.tolist(),
    "numerical_features": num_cols,
    "binary_features": ["SeniorCitizen"],
    "target": "Churn",
    "target_mapping": {"No": 0, "Yes": 1},
    "training_rows": len(X_train_final),
    "testing_rows": len(X_test_final),
    "clustering_rows": len(X_cluster)
}

config = {
    "dataset": "Dataset_ATS_v2.csv",
    "train_test_split": 0.20,
    "random_state": 42,
    "stratified_by": "Churn",
    "categorical_encoding": "OneHotEncoder",
    "unknown_categories": "ignore",
    "numerical_scaling": "StandardScaler",
    "scaled_features": num_cols,
    "predictive_scaler_fit": "training data only",
    "clustering_scaler_fit": "full clustering dataset",
    "records_removed": 0
}

for filename, content in [
    ("feature_information.json", feature_info),
    ("preprocessing_config.json", config)
]:
    path = os.path.join("Data_Preparation", filename)

    with open(path, "w") as f:
        json.dump(content, f, indent=4)

    shutil.copy2(path, destination)
    print(filename, "- saved and backed up")

print("\nTotal files:",
      len(os.listdir("Data_Preparation")))


feature_information.json - saved and backed up
preprocessing_config.json - saved and backed up

Total files: 9


In [14]:

# Step 14: Creating the GitHub README
# I documented my Data Engineer responsibilities,
# preprocessing workflow, datasets, and handover files.

readme = """# ACS WIL – Customer Churn Data Preprocessing

## Overview
This folder contains the data preprocessing and feature
preparation deliverables for the ACS WIL Customer Churn
Analysis project.

## My Role: Data Engineer
I prepared the customer churn dataset for the team members
working on ANN classification and K-Means clustering.

## Dataset
- Source: Dataset_ATS_v2.csv
- Total records: 7,043
- Original columns: 10
- Input features: 9
- Target: Churn

## Preprocessing Workflow
1. Inspected dataset structure and data types.
2. Checked missing, blank, duplicate-looking, and invalid values.
3. Identified conflicting phone-service records.
4. Retained all records because corrections could not be verified.
5. Separated input features and the Churn target.
6. Created an 80/20 stratified train-test split (random_state=42).
7. Applied one-hot encoding to categorical features.
8. Standardized tenure and MonthlyCharges.
9. Prepared separate predictive and clustering datasets.
10. Saved fitted preprocessing objects for reproducibility.

## Prepared Datasets
| File | Rows | Columns | Purpose |
|---|---:|---:|---|
| train_preprocessed.csv | 5,634 | 17 | ANN training |
| test_preprocessed.csv | 1,409 | 17 | ANN testing |
| clustering_preprocessed.csv | 7,043 | 16 | K-Means input |

The predictive datasets contain 16 features and Churn
encoded as No=0 and Yes=1.

The clustering dataset excludes Churn.

## Data Quality Findings
- Missing values: 0
- Blank text values: 0
- Duplicate-looking rows: 302
- Zero-tenure records: 11
- Conflicting phone-service records: 260
- Records removed: 0

Duplicate-looking rows were retained because no unique
customer identifier was available. Conflicting records
were documented rather than corrected without evidence.

## Additional Files
- encoder.pkl: fitted categorical encoder
- predictive_scaler.pkl: training-fitted numerical scaler
- clustering_scaler.pkl: full-data clustering scaler
- feature_information.json: feature names and target mapping
- preprocessing_config.json: reproducibility settings
- data_quality_report.csv: data-quality findings

## Important Notes
- The predictive encoder and scaler were fitted only on
  training data to avoid test-data leakage during fitting.
- The clustering dataset uses all available customers.
- The clustering teammate should review feature weighting,
  as one-hot features remain binary.
- No ANN or K-Means models were trained in this notebook.
"""

readme_path = "Data_Preparation/README.md"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

shutil.copy2(readme_path, destination)

print("README created and backed up successfully!")
print("Total files:", len(os.listdir("Data_Preparation")))


README created and backed up successfully!
Total files: 10


In [15]:

# Step 15: Final Deliverable Verification
# I verified that all 10 deliverable files exist,
# match their Google Drive backups, and contain
# the expected preprocessed datasets.

import hashlib

folder = "Data_Preparation"

expected_files = [
    "train_preprocessed.csv",
    "test_preprocessed.csv",
    "clustering_preprocessed.csv",
    "encoder.pkl",
    "predictive_scaler.pkl",
    "clustering_scaler.pkl",
    "data_quality_report.csv",
    "feature_information.json",
    "preprocessing_config.json",
    "README.md"
]

all_valid = True

for filename in expected_files:
    local_path = os.path.join(folder, filename)
    drive_path = os.path.join(destination, filename)

    valid = (
        os.path.isfile(local_path)
        and os.path.isfile(drive_path)
        and os.path.getsize(local_path) > 0
    )

    if valid:
        with open(local_path, "rb") as f:
            local_hash = hashlib.sha256(f.read()).hexdigest()
        with open(drive_path, "rb") as f:
            drive_hash = hashlib.sha256(f.read()).hexdigest()
        valid = local_hash == drive_hash

    all_valid = all_valid and valid
    print(filename, "-", "PASS" if valid else "FAIL")

train_check = pd.read_csv(os.path.join(folder, "train_preprocessed.csv"))
test_check = pd.read_csv(os.path.join(folder, "test_preprocessed.csv"))
cluster_check = pd.read_csv(os.path.join(folder, "clustering_preprocessed.csv"))

shapes_valid = (
    train_check.shape == (5634, 17)
    and test_check.shape == (1409, 17)
    and cluster_check.shape == (7043, 16)
    and "Churn" not in cluster_check.columns
    and train_check.columns.equals(test_check.columns)
    and not train_check.isna().any().any()
    and not test_check.isna().any().any()
    and not cluster_check.isna().any().any()
)

print("\nDataset validation:", "PASS" if shapes_valid else "FAIL")
print("Google Drive backup:", "PASS" if all_valid else "FAIL")
print("Overall:", "PASS" if all_valid and shapes_valid else "FAIL")


train_preprocessed.csv - PASS
test_preprocessed.csv - PASS
clustering_preprocessed.csv - PASS
encoder.pkl - PASS
predictive_scaler.pkl - PASS
clustering_scaler.pkl - PASS
data_quality_report.csv - PASS
feature_information.json - PASS
preprocessing_config.json - PASS
README.md - PASS

Dataset validation: PASS
Google Drive backup: PASS
Overall: PASS


In [16]:

# Step 16: Packaging My Data Engineer Deliverables
# I created a ZIP archive containing all 10 verified
# preprocessing files and saved it to Google Drive.

import shutil
import zipfile
import os

assert all_valid and shapes_valid, "Verification must pass first."

zip_base = "/content/drive/MyDrive/ACS_WIL/ACS_Data_Preparation"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/content",
    base_dir="Data_Preparation"
)

with zipfile.ZipFile(zip_path, "r") as archive:
    files_inside = archive.namelist()
    archived_files = [
        name for name in files_inside
        if not name.endswith("/")
    ]
    zip_valid = (
        archive.testzip() is None
        and len(archived_files) == 10
        and all(
            f"Data_Preparation/{name}" in archived_files
            for name in expected_files
        )
    )

print("ZIP created:", zip_path)
print("Files inside:", len(archived_files))
print("ZIP size:", os.path.getsize(zip_path), "bytes")
print("ZIP verification:", "PASS" if zip_valid else "FAIL")


ZIP created: /content/drive/MyDrive/ACS_WIL/ACS_Data_Preparation.zip
Files inside: 10
ZIP size: 119685 bytes
ZIP verification: PASS
